In [ ]:
! pip install transformers pandas datasets scikit-learn

The specific model we're going to use is ESM-2. The citation for this model is [Lin et al, 2022](https://www.biorxiv.org/content/10.1101/2022.07.20.500902v1).

There are several ESM-2 checkpoints with differing model sizes. Larger models will generally have better accuracy, but they require more GPU memory and will take much longer to train. The available ESM-2 checkpoints are:

| Checkpoint name | Num layers | Num parameters |
|------------------------------|----|----------|
| `esm2_t48_15B_UR50D`         | 48 | 15B     |
| `esm2_t36_3B_UR50D`          | 36 | 3B      |
| `esm2_t33_650M_UR50D`        | 33 | 650M    |
| `esm2_t30_150M_UR50D`        | 30 | 150M    |
| `esm2_t12_35M_UR50D`         | 12 | 35M     |
| `esm2_t6_8M_UR50D`           | 6  | 8M      |

Note that the larger checkpoints may be very difficult to train without a large cloud GPU like an A100 or H100, and the largest 15B parameter checkpoint will probably be impossible to train on **any** single GPU! Also, note that memory usage for attention during training will scale as `O(batch_size * num_layers * seq_len^2)`, so larger models on long sequences will use quite a lot of memory! We will use the `esm2_t12_35M_UR50D` checkpoint for this notebook, which should train on any Colab instance or modern GPU.

In [ ]:
# model_checkpoint = "facebook/esm2_t12_35M_UR50D"
# model_checkpoint = "facebook/esm2_t6_8M_UR50D"
model_checkpoint = "esm_ecm"

## Data preparation

In [ ]:
import pandas as pd

def read_data(filename, target):

  file_path = filename + '.tsv.gz'

  df = pd.read_csv(file_path, compression='gzip', sep='\t')

  df = df.dropna(subset=['Gene Ontology (GO)'])
  df = df.dropna(subset=['Sequence'])
  df = df[~df["Sequence"].duplicated()]
  df = df[df['Sequence'].str.len() <= 1800]

  df.drop(['Entry', 'Gene Ontology (GO)'], axis=1, inplace=True)
  df['target'] = target

  df.reset_index(drop=True, inplace=True)

  return df

In [ ]:
import pandas

dev_df_positive = read_data('Extracellular_matrix_organization', 1)
dev_df_negative = read_data('Not_extracellular_matrix_organization', 0)

print(dev_df_positive.shape)
print(dev_df_negative.shape)

(40155, 2)
(46427, 2)


In [ ]:
# third time
dev_df = pd.concat([dev_df_positive.iloc[20000:30000], dev_df_negative.iloc[20000:30000]])
dev_df.head()

,Sequence,target
20000,MREPRPWGRCVGAILTSVFFLLGCWGLSDFQQQFLQALDPEEVTSY...,1
20001,MCPRNRHVNSLLLSSACLASLTPSSSPSPSSLVLLVLQQFLQALDP...,1
20002,MFSVCWPRSITTVHLQDTILTYCTKGRQRRRSTVAMATWTPADVLP...,1
20003,MRGWAAGLLGAEVLAGLTPLPVAGRPLGRGAERILAVPVRTDAQGR...,1
20004,MDALDVLAGLTPLPVAGRPLGRGAERILAVPVRTDAQGRLVSHVVS...,1


In [ ]:
sequences = dev_df["Sequence"].tolist()
labels = dev_df["target"].tolist()
len(sequences) == len(labels)

True

## Splitting the data

In [ ]:
from sklearn.model_selection import train_test_split

train_sequences, test_sequences, train_labels, test_labels = train_test_split(sequences, labels, test_size=0.4, shuffle=True)

## Tokenizing the data

In [ ]:
from transformers import AutoTokenizer

# load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [ ]:
# tokenizer two dataset
train_tokenized = tokenizer(train_sequences)
test_tokenized = tokenizer(test_sequences)

## Dataset creation

In [ ]:
from datasets import Dataset

# build datasets

train_dataset = Dataset.from_dict(train_tokenized)
test_dataset = Dataset.from_dict(test_tokenized)

train_dataset = train_dataset.add_column('labels', train_labels)
test_dataset = test_dataset.add_column('labels', test_labels)

train_dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 12000
})

## Model loading

In [ ]:
from transformers import TFAutoModelForSequenceClassification
# load model
model = TFAutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=1)

All model checkpoint layers were used when initializing TFEsmForSequenceClassification.

All the layers of TFEsmForSequenceClassification were initialized from the model checkpoint at drive/MyDrive/esm_ecm.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFEsmForSequenceClassification for predictions without further training.


In [ ]:
# prepare dataset
tf_train_set = model.prepare_tf_dataset(
    train_dataset,
    batch_size=4,
    shuffle=True,
    tokenizer=tokenizer
)

tf_test_set = model.prepare_tf_dataset(
    test_dataset,
    batch_size=4,
    shuffle=False,
    tokenizer=tokenizer
)

In [ ]:
from transformers import AdamWeightDecay
from keras.metrics import Precision, Recall

# build model
model.compile(
    optimizer=AdamWeightDecay(2e-5),
    metrics=['accuracy', Precision(), Recall()]
)

In [ ]:
# training
# first time 3 epochs, second, third, fourth time 1 epochs

model.fit(tf_train_set, validation_data=tf_test_set, epochs=1)

3000/3000 [==============================] - 1346s 440ms/step - loss: 0.0183 - accuracy: 0.9778 - precision_1: 0.9835 - recall_1: 0.9611 - val_loss: 0.0120 - val_accuracy: 0.9872 - val_precision_1: 0.9865 - val_recall_1: 0.9730


In [ ]:
# save model and tokenizer
model.save_pretrained('esm_ecm')
tokenizer.save_pretrained('esm_ecm')